# 07_行政_部門月費用

NumPy 二維陣列與 axis

任務：只把三個數值欄轉成 NumPy 陣列，說明 shape。；用 axis=1 算各部門季度總額。；用 axis=0 算各月總額，核對兩種加總的總計。

規則：每列一個部門、每欄一個月份；費用可跨月加總。部門名稱不能放進數值陣列。

先人工核對再執行。資料皆為教學模擬值。

In [ ]:
%pip -q install openpyxl numpy matplotlib

In [ ]:
from pathlib import Path
import csv
from openpyxl import load_workbook

# 本機：把 Notebook 與 Excel 放在同一資料夾。Colab：先上傳本例 Excel。
SOURCE = Path('07_行政_部門月費用.xlsx')
if not SOURCE.exists():
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError(f'請將 {SOURCE.name} 放到目前工作目錄：{Path.cwd()}')
    files.upload()
if not SOURCE.exists():
    raise FileNotFoundError(f'尚未找到 {SOURCE.name}，請確認上傳檔名')
OUT = Path('output') / '07_行政_部門月費用'
OUT.mkdir(parents=True, exist_ok=True)

def read_rows(sheet):
    data = sheet.iter_rows(values_only=True)
    headers = next(data)
    return [dict(zip(headers, row)) for row in data if any(v is not None for v in row)]

def save_csv(name, records):
    if not records:
        print('沒有符合條件的資料，未輸出 CSV')
        return
    with (OUT / name).open('w', newline='', encoding='utf-8-sig') as f:
        writer = csv.DictWriter(f, fieldnames=list(records[0]))
        writer.writeheader()
        writer.writerows(records)

# 來源檔只有原始值，沒有需要重算的 Excel 公式。
wb = load_workbook(SOURCE, read_only=True, data_only=True)
print('工作表：', wb.sheetnames)
rows = read_rows(wb['資料'])
print('第一筆：', rows[0])

In [ ]:
import numpy as np
a = np.array([[r[k] for k in ['1月費用_元','2月費用_元','3月費用_元']] for r in rows], dtype=float)
print('shape：', a.shape)
print('各部門：', a.sum(axis=1))
print('各月：', a.sum(axis=0))
assert a.shape == (3,3)
assert a.sum(axis=1).tolist() == [36000,30000,54000]
assert a.sum(axis=0).tolist() == [38000,38000,44000]
assert a.sum() == 120000
wb.close()

## 驗收答案

shape=(3,3)；部門總額 36,000、30,000、54,000；月總額 38,000、38,000、44,000。